# 面试问题：ZeRO-1/2/3 分别切什么？显存、通信、参数生命周期和 checkpoint 怎样实现？

**一句话回答**：普通数据并行在每卡重复参数、梯度和优化器状态；ZeRO-1 只切 optimizer states，ZeRO-2 再切 gradients，ZeRO-3 连 parameters 也切。节省显存的代价是 reduce-scatter/all-gather、参数预取、碎片管理和分片 checkpoint 复杂度，不能只背三个阶段。

本 Notebook 从字节公式开始，实现连续分片、reduce-scatter、all-gather、Stage-3 参数物化/释放、offload 代价与跨 DP world size 恢复。


In [ ]:
from dataclasses import dataclass
import hashlib,math
import numpy as np

SEED129=12901; rng129=np.random.default_rng(SEED129)
assert SEED129==12901
assert 2+2+8==12
assert np.isfinite(rng129.normal())


## 1. 先把每参数字节拆开

常见混合精度 Adam 口径可近似为 BF16 参数 2B、BF16/FP32 梯度若干、FP32 master weight 4B、两个 FP32 moment 共 8B。实现差异会改变数字，所以面试时应列组件再求和。激活、通信 bucket 和临时 all-gather buffer 需另算。


In [ ]:
@dataclass(frozen=True)
class StateBytes129:
    param:int=2; grad:int=2; master:int=4; moments:int=8
    @property
    def optimizer(self): return self.master+self.moments
    @property
    def total(self): return self.param+self.grad+self.optimizer
sb129=StateBytes129()
assert sb129.optimizer==12
assert sb129.total==16
assert 7e9*sb129.total/1e9==112


## 2. 三个 Stage 的稳态模型状态公式

DP world size 为 `P` 时，Stage-1 每卡约 `param+grad+optimizer/P`；Stage-2 为 `param+(grad+optimizer)/P`；Stage-3 为全部 `/P`。这不包含峰值临时参数、激活和 allocator 碎片，因此只能作为下界与 sanity check。


In [ ]:
def state_bytes129(N,P,stage,s=StateBytes129()):
    factors={0:s.total,1:s.param+s.grad+s.optimizer/P,2:s.param+(s.grad+s.optimizer)/P,3:s.total/P}
    return N*factors[stage]
vals129=[state_bytes129(7e9,8,s)/1e9 for s in range(4)]
assert vals129[0]>vals129[1]>vals129[2]>vals129[3]
assert math.isclose(vals129[3],14.0)
assert state_bytes129(7e9,16,3)<state_bytes129(7e9,8,3)


## 3. 分片必须定义 padding、owner 与重组顺序

参数展平后补齐到 world size 的整数倍，再按连续区间分配 owner。元数据要保存原始 shape、dtype、offset 和 padding；否则从 8 卡恢复到 4 卡时无法无损重组。这里手写 shard/gather，并验证非整除长度。


In [ ]:
def shard129(x,P):
    n=len(x); width=math.ceil(n/P); padded=np.pad(np.asarray(x),(0,width*P-n)); return [padded[i*width:(i+1)*width].copy() for i in range(P)],n
def gather129(parts,n): return np.concatenate(parts)[:n]
vec129=np.arange(11,dtype=float); parts129,n129=shard129(vec129,4)
assert [len(x) for x in parts129]==[3,3,3,3]
assert np.array_equal(gather129(parts129,n129),vec129)
assert parts129[-1][-1]==0


## 4. Stage-2/3 用 reduce-scatter 让每个 rank 只保留自己的梯度

所有 rank 的局部梯度先按元素求和/平均，再只返回 owner shard；更新参数前后按需要 all-gather。生产 collective 会分 bucket 并与 backward overlap，这里显式计算 oracle，强调平均的除数是参与样本/token，而不一定只是 rank 数。


In [ ]:
def reduce_scatter129(local_grads):
    mean=np.mean(np.stack(local_grads),axis=0); return shard129(mean,len(local_grads))[0]
locals129=[np.arange(7,dtype=float)+i for i in range(3)]; rs129=reduce_scatter129(locals129)
expected129=np.mean(np.stack(locals129),axis=0)
assert np.allclose(gather129(rs129,len(expected129)),expected129)
assert len(rs129)==3
assert all(len(x)==3 for x in rs129)


## 5. Stage-3 参数是按层临时物化的

forward 到某层前预取所有参数 shard 并 all-gather，计算完成后在安全点释放；backward 还需再次物化。预取太晚会产生气泡，太早会推高峰值显存。状态机必须防止层仍有计算引用时释放，也要限制同时驻留窗口。


In [ ]:
class Materializer129:
    def __init__(self,limit): self.limit=limit; self.live=[]; self.events=[]
    def acquire(self,layer):
        if layer not in self.live:
            if len(self.live)>=self.limit: self.release(self.live[0])
            self.live.append(layer); self.events.append(("gather",layer))
    def release(self,layer):
        if layer in self.live: self.live.remove(layer); self.events.append(("release",layer))
mat129=Materializer129(2); [mat129.acquire(x) for x in [0,1,2]]
assert mat129.live==[1,2]
assert mat129.events[2]==("release",0)
assert sum(e[0]=="gather" for e in mat129.events)==3


## 6. Loss scaling 发生在分片前后都必须语义一致

混合精度下先对 scaled gradient 做 collective，再统一检测 inf/nan；任何 rank overflow 都要让所有 rank 跳过同一次 optimizer step。更新 FP32 master shard 后再生成低精度参数 shard，不能让部分 rank 单独前进一步。


In [ ]:
def global_overflow129(shards): return any(not np.all(np.isfinite(x)) for x in shards)
good129=[np.array([1.,2.]),np.array([3.])]; bad129=[good129[0],np.array([np.inf])]
assert not global_overflow129(good129)
assert global_overflow129(bad129)
master129=np.array([1.,2.],dtype=np.float32); master129-=.1*np.array([.5,-.5],dtype=np.float32)
assert np.allclose(master129,[.95,2.05])


## 7. Offload 是否划算取决于能否隐藏传输

CPU/NVMe offload 扩大容量，却把 PCIe、内存带宽和 page fault 变成关键路径。用 `bytes/bandwidth` 与该层计算时间比较；若传输无法被预取覆盖，吞吐会明显下降。还应区分 steady-state 带宽与尾延迟，不用峰值宣传值做规划。


In [ ]:
def hidden_fraction129(bytes_,bandwidth_gbps,compute_ms):
    transfer_ms=bytes_/(bandwidth_gbps*1e9)*1e3
    return min(1.0,compute_ms/transfer_ms),transfer_ms
hidden129,t129=hidden_fraction129(2e9,24,40)
assert 0<hidden129<1
assert t129>40
assert hidden_fraction129(2e9,100,40)[0]==1.0


## 8. Checkpoint 是全局一致快照，不是若干 shard 文件

manifest 要记录 step、world size、展平顺序、shape/dtype、参数/optimizer/RNG shard 和完成标记。写入临时版本，所有 rank 成功后原子发布。恢复到新 DP 数量时先按 manifest 重组逻辑向量，再重新分片；缺 shard 或 digest 不符必须失败关闭。


In [ ]:
old129,_=shard129(np.arange(13),4); full129=gather129(old129,13); new129,nnew129=shard129(full129,2)
manifest129={"step":500,"old_dp":4,"new_dp":2,"logical_n":nnew129,"digest":hashlib.sha256(full129.tobytes()).hexdigest()}
assert np.array_equal(gather129(new129,nnew129),np.arange(13))
assert len(new129)==2 and manifest129["logical_n"]==13
assert len(manifest129["digest"])==64


## 面试总结

回答顺序建议是：**列每参数字节 → 推 Stage-0/1/2/3 稳态下界 → 解释 padding/owner → reduce-scatter 梯度 → all-gather 参数 → Stage-3 预取/释放窗口 → 全局 overflow → offload 带宽覆盖 → 两阶段 checkpoint 与 reshard**。ZeRO 用更多生命周期和通信复杂度换显存容量，选择 Stage 要由真实激活峰值、网络和吞吐决定。

延伸阅读：[ZeRO](https://arxiv.org/abs/1910.02054)、[ZeRO-Infinity](https://arxiv.org/abs/2104.07857)、[Megatron 3D Parallelism](https://arxiv.org/abs/2104.04473)。
